# Lektion 1: Datenbankzugriff mit Python

Übungsnotebook zu Lektion 1 · Hilfsmittel: Skript Lektion 1

In dieser Lektion schreiben Sie den Datenbankzugriff selbst, ohne die Hilfsfunktion `sql()`.
Sie arbeiten mit einer **Kopie** der Datenbank (`uebung_l01.db`). Dort dürfen Sie Daten einfügen und ändern,
ohne die Datenbank für die anderen Lektionen zu verändern. Wenn etwas schiefgeht: Vorbereitungszelle
erneut ausführen, dann ist die Kopie wieder im Ausgangszustand.

## Vorbereitung

Führen Sie zuerst diese Zelle aus (Umschalt + Enter). Sie legt die Übungskopie der Datenbank an.

In [ ]:
import sqlite3
import shutil
import pandas as pd

# Übungskopie anlegen: hier dürfen Sie alles ändern
shutil.copy("../daten/shop_oltp.db", "uebung_l01.db")
print("Übungskopie uebung_l01.db angelegt")

---
### Aufgabe 1: Verbindung und Cursor

a) Bauen Sie eine Verbindung `con` zur Datei `uebung_l01.db` auf und erzeugen Sie einen Cursor `cur`.

b) Ermitteln Sie mit dem Cursor die Anzahl der Kunden (`SELECT COUNT(*) FROM kunde`) und geben Sie **nur die Zahl** aus, nicht das Tupel.

---
### Aufgabe 2: fetchone, fetchmany, fetchall

Führen Sie `SELECT name FROM kategorie ORDER BY name` aus. Holen Sie dann nacheinander ab:

1. die erste Zeile mit `fetchone()`,
2. die nächsten zwei Zeilen mit `fetchmany(2)`,
3. alle restlichen mit `fetchall()`.

Geben Sie jedes Ergebnis aus. Wie viele Zeilen liefert `fetchall()` und warum nicht alle sechs?

*Ihre Antwort:*

---
### Aufgabe 3: Spaltennamen und Schleife

a) Führen Sie `SELECT * FROM artikel` aus und geben Sie mit `cur.description` eine Liste aller **Spaltennamen** aus.

b) Geben Sie alle Artikel mit einem Listenpreis über 500 € in der Form `Notebook ProBook 14: 899.0 €` aus. Laufen Sie dazu in einer for-Schleife über den Cursor.

---
### Aufgabe 4: Eine Funktion mit Platzhalter

Schreiben Sie eine Funktion `kunden_in_ort(ort)`, die Vor- und Nachname aller Kunden eines Ortes als Liste zurückgibt.
Verwenden Sie einen **Platzhalter** für den Ortsnamen.

Testen Sie die Funktion mit `"Köln"` und `"Hamburg"`.

---
### Aufgabe 5: SQL-Injection selbst ausprobieren

Die Funktion unten sucht Kunden nach Nachnamen. Sie baut die Eingabe per f-String in den SQL-Text ein.

a) Rufen Sie sie mit `"Koch"` auf. Wie viele Kunden werden gefunden?
b) Rufen Sie sie mit `"x' OR '1'='1"` auf. Wie viele Kunden werden jetzt gefunden? Lassen Sie sich auch den erzeugten SQL-Text ausgeben.
c) Schreiben Sie eine sichere Funktion `kunde_suchen_sicher(nachname)` mit Platzhalter und testen Sie sie mit beiden Eingaben.

In [ ]:
def kunde_suchen_unsicher(nachname):
    abfrage = f"SELECT vorname, nachname, email FROM kunde WHERE nachname = '{nachname}'"
    print("SQL:", abfrage)
    return con.execute(abfrage).fetchall()

*Ihre Antwort:*

---
### Aufgabe 6: Daten einfügen: lastrowid und commit

Nordlicht Versand nimmt Drucker ins Sortiment.

a) Legen Sie die Kategorie „Drucker“ an. Geben Sie mit `cur.lastrowid` die neue `kategorie_id` aus.
b) Legen Sie in dieser Kategorie den Artikel „Laserdrucker LX 200“ für 189,00 € an. Verwenden Sie für alle Werte Platzhalter und die ID aus a).
c) Schreiben Sie die Änderungen fest und prüfen Sie mit einer Abfrage, dass der Artikel mit Kategorienamen gespeichert ist.

---
### Aufgabe 7: Was passiert ohne commit?

Die Zelle unten öffnet eine **zweite Verbindung** `con2` zur selben Datenbank, so als würde ein Kollege gleichzeitig darauf zugreifen.

1. Fügen Sie über `con` einen neuen Ort ein (PLZ `06108`, Name `Halle (Saale)`, Region 2), aber **ohne** commit.
2. Zählen Sie über `con` und über `con2` die Orte. Was fällt auf?
3. Führen Sie `con.commit()` aus und zählen Sie über `con2` erneut.

In [ ]:
con2 = sqlite3.connect("uebung_l01.db")
print("zweite Verbindung geöffnet")

*Ihre Antwort:*

---
### Aufgabe 8: Transaktion mit rollback

Kundin 5 bestellt per Telefon zwei Artikel. Die Bestellung soll **ganz oder gar nicht** gespeichert werden.

a) Speichern Sie Bestellung 321 (Kunde 5, Mitarbeiter 3, Bestelldatum 2025-12-15, kein Lieferdatum, Status „offen“) mit zwei Positionen:
Position 1: Artikel 1, Menge 1, Einzelpreis 899.00 · Position 2: Artikel 4, Menge 2, Einzelpreis 159.00.
Bauen Sie absichtlich einen Fehler ein: Geben Sie beiden Positionen die Positionsnummer 1.
Fangen Sie den Fehler mit `try` / `except sqlite3.IntegrityError` ab und rufen Sie dann `rollback()` auf.

b) Prüfen Sie: Gibt es Bestellung 321 in der Datenbank?

c) Korrigieren Sie den Fehler und speichern Sie die Bestellung erfolgreich.

*Ihre Antwort:*

---
### Aufgabe 9: Viele Datensätze mit executemany

Der Vertrieb liefert eine Liste neuer Liefergebiete. Fügen Sie alle mit **einem** Aufruf von `executemany()` in die Tabelle `ort` ein, schreiben Sie fest und geben Sie `cur.rowcount` aus.
Wie viele Orte gibt es danach insgesamt?

In [ ]:
neue_orte = [
    ("99084", "Erfurt", 2),
    ("66111", "Saarbrücken", 4),
    ("30159", "Hannover", 1),
    ("60311", "Frankfurt am Main", 4),
    ("93047", "Regensburg", 3),
]

*Ihre Antwort:*

---
### Aufgabe 10: pandas mit Parametern

a) Ermitteln Sie mit `pd.read_sql_query()` den Umsatz je Jahr (`strftime('%Y', bestelldatum)`) für **Kunde 23**, ohne stornierte Bestellungen. Übergeben Sie die Kunden-ID als Parameter.

b) In den anderen Notebooks gibt es die Hilfsfunktion `sql()`. Erklären Sie in zwei bis drei Sätzen, was sie tut.

*Ihre Antwort:*

---
### Aufgabe 11: Prüfungsaufgabe: Code beurteilen

In einer Webanwendung finden Sie folgenden Java-Code für die Anmeldung:

```java
String sql = "SELECT * FROM benutzer WHERE login = '" + login
           + "' AND passwort_hash = '" + hash + "'";
ResultSet rs = statement.executeQuery(sql);
```

a) Benennen Sie die Sicherheitslücke und beschreiben Sie einen möglichen Angriff.
b) Wie heißt die wirksamste Gegenmaßnahme? Schreiben Sie die ersten beiden Zeilen entsprechend um.
c) Ein Kollege schlägt vor, auch den Tabellennamen über einen Platzhalter zu übergeben: `SELECT * FROM ? WHERE ...`. Ist das möglich?

Schließen Sie zum Abschluss Ihre Verbindung mit `con.close()`.

*Ihre Antwort:*